In [1]:
import numpy as np
import cv2

In [3]:
def findDist(pts1, pts2):
    return ((pts1[0] - pts2[0]) ** 2 + (pts1[1] - pts2[1]) ** 2) ** 0.5

In [2]:
def reorder(points):
    # print(points.shape)
    
    newPoints = np.zeros_like(points)
    points = points.reshape((4,2))
    
    
    add = points.sum(1)
    newPoints[0] = points[np.argmin(add)]
    newPoints[3] = points[np.argmax(add)]

    diff = np.diff(points, axis = 1)
    newPoints[1] = points[np.argmin(diff)]
    newPoints[2] = points[np.argmax(diff)]

    return newPoints

In [3]:
def getWarp(img, contPoints, w, h):
    points = reorder(contPoints)
    source_points = np.float32(points)
    dest_points = np.float32([[0,0], [w,0], [0,h], [w,h]])
    matrix = cv2.getPerspectiveTransform(source_points, dest_points)
    img_out = cv2.warpPerspective(img, matrix, (w, h))
    
    # source_points = source_points.reshape(4,2)   
    # min_x = np.min(source_points[:, 0])
    # max_x = np.max(source_points[:, 0])
    # min_y = np.min(source_points[:, 1])
    # max_y = np.max(source_points[:, 1])
    
    # # Calculate width and height
    # img_width = max_x - min_x
    # img_height = max_y - min_y
    
    cropped_img = img_out[20:img_out.shape[0]-10, 20:img_out.shape[1]-10]
    return cropped_img
    
    # return cropped_img, int(img_width), int(img_height)

In [4]:
def Contour_Corner_Detect(img, imgEro, minArea, filter, drawCont):
    contour, hire = cv2.findContours(imgEro, mode = cv2.RETR_EXTERNAL, method = cv2.CHAIN_APPROX_SIMPLE)
    finalContours = []
    for i in contour:
        area = cv2.contourArea(i)
        if area > minArea:
            peri = cv2.arcLength(i, True)
            approx = cv2.approxPolyDP(i, .02*peri, True)
            bbox = cv2.boundingRect(approx)

            if filter>0:
                if len(approx) == filter:
                    finalContours.append([len(approx), approx, area, bbox, i])
            else:
                finalContours.append([len(approx), approx, area, bbox, i])

    finalContours = sorted(finalContours, key = lambda x:x[2], reverse = True)

    if drawCont:
        for con in finalContours:
            cv2.drawContours(img, con[4], -1, color = (255, 0 ,255), thickness = 4)

    return img, finalContours

In [5]:
def grey_edgeDetect(img, cTh = [100,100], imgShow = False, ccd = False, minArea = 1000, filter = 0, drawCont = False):
    imgGrey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    imgBlur = cv2.GaussianBlur(imgGrey, (5,5), 1)
    imgCEdge = cv2.Canny(imgBlur, cTh[0], cTh[1])

    imgDil = cv2.dilate(imgCEdge, np.ones((3,3)), iterations = 2)
    imgEro = cv2.erode(imgDil, np.ones((3,3)), iterations = 1)

    if imgShow: 
        cv2.imshow('cannyEdge', imgEro) 
        cv2.waitKey(0)
    
    if ccd: contourImg, finalContous = Contour_Corner_Detect(img, imgEro, minArea, filter, drawCont)

    return contourImg, finalContous